# Проект: FashionMNIST

Свой сквозной проект параллельно проекту автора на MNIST. После каждого видео сюда добавляется новый раздел: данные → модель → цикл обучения → scheduler → сохранение → EarlyStopping → финальный test.

| у автора | здесь |
|:---|:---|
| MNIST: цифры 0–9 | FashionMNIST: 10 видов одежды |
| 60 000 train / 10 000 test, 28×28, серые | то же самое |

Формы совпадают, поэтому видео подходит один в один. Классы другие: результат и ошибки сети будут свои.

**Правило проекта:** ячейки идут сверху вниз, каждый раздел опирается на код прошлых. Перед новым разделом перезапусти ноутбук целиком (Restart → Run All до нового раздела).

Решения — в `fashion_mnist_решения.ipynb`, только после своей попытки.

In [ ]:
import gzip
import os
import shutil
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split, RandomSampler, SequentialSampler

torch.manual_seed(0)


def need(cond, msg):
    assert cond, msg


def need_tensor(x, name):
    assert isinstance(x, torch.Tensor), f"{name} — пока не тензор (замени ... на код)"

---
# Видео 5. Данные → `Dataset` → `DataLoader`

У автора в этом видео: картинки по папкам-классам → свой `MnistDataset` → `random_split` → три `DataLoader`. Здесь то же самое на FashionMNIST.

## 5.0 Подготовка данных

Автор раскладывает MNIST по папкам заранее, вне видео. Ячейка ниже делает это для FashionMNIST: качает архивы (~30 МБ) и сохраняет 70 000 png в `_data/fashion/{train,test}/<класс>/`. Первый запуск — 1–3 минуты, потом ячейка видит готовые папки и ничего не делает.

Имена папок — названия одежды (`t-shirt`, `ankle-boot`, …), а не цифры: как у автора, «более общий случай».

In [ ]:
DATA = "_data"
RAW = os.path.join(DATA, "raw")
FASHION = os.path.join(DATA, "fashion")
TRAIN_DIR = os.path.join(FASHION, "train")
TEST_DIR = os.path.join(FASHION, "test")

URL = "https://github.com/zalandoresearch/fashion-mnist/raw/master/data/fashion/"
FILES = {
    "train": ("train-images-idx3-ubyte.gz", "train-labels-idx1-ubyte.gz"),
    "test": ("t10k-images-idx3-ubyte.gz", "t10k-labels-idx1-ubyte.gz"),
}
CLASS_NAMES = ["t-shirt", "trouser", "pullover", "dress", "coat",
               "sandal", "shirt", "sneaker", "bag", "ankle-boot"]


def download(file_name):
    path = os.path.join(RAW, file_name)
    if not os.path.exists(path):
        os.makedirs(RAW, exist_ok=True)
        urllib.request.urlretrieve(URL + file_name, path)
    return path


def read_images(path):
    with gzip.open(path, "rb") as f:
        return np.frombuffer(f.read(), dtype=np.uint8, offset=16).reshape(-1, 28, 28)


def read_labels(path):
    with gzip.open(path, "rb") as f:
        return np.frombuffer(f.read(), dtype=np.uint8, offset=8)


def prepare_split(split):
    root = os.path.join(FASHION, split)
    if os.path.isdir(root):
        print(split, "— уже готов")
        return
    tmp = root + "_tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    images_file, labels_file = FILES[split]
    images = read_images(download(images_file))
    labels = read_labels(download(labels_file))
    for name in CLASS_NAMES:
        os.makedirs(os.path.join(tmp, name))
    for k, (img, label) in enumerate(zip(images, labels)):
        Image.fromarray(img).save(os.path.join(tmp, CLASS_NAMES[label], f"{split}_{k:05d}.png"))
    os.rename(tmp, root)
    print(split, "—", len(images), "картинок")


prepare_split("train")
prepare_split("test")
print(sorted(os.listdir(TRAIN_DIR)))

## 5.1 Свой `ImageDataset`
*видео 5: 2:36–9:56*

Возьми свой `FruitsDataset` из `05` и сделай из него `ImageDataset` для этого проекта. Требования те же:

| что | требование |
|:---|:---|
| `self.classes` | отсортированные имена папок-классов |
| `self.class_to_idx` | имя → индекс |
| `self.data_list` | пары `(путь, индекс)`, только `.png` |
| `__getitem__` | `(x, label)`: `x` — `float32` `(1, 28, 28)` в `0..1`, `label` — `int` |
| `transform` | если задан — применить к `x` |

**Новое:** `self.targets` — список меток всех примеров в том же порядке, что `data_list`. Так можно считать баланс классов, не открывая 60 000 файлов (у `ImageFolder` из видео 6 такое поле тоже есть).

**Ожидаемый вывод:**

```text
['ankle-boot', 'bag', 'coat', 'dress', 'pullover', 'sandal', 'shirt', 'sneaker', 't-shirt', 'trouser']
60000 10000 torch.Size([1, 28, 28]) torch.float32 0
```

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.classes = sorted(
            name for name in os.listdir(root) if os.path.isdir(os.path.join(root, name))
        )
        self.class_to_idx = {name: i for i, name in enumerate(self.classes)}
        self.data_list = []
        for name in self.classes:
            folder = os.path.join(root, name)
            for file_name in sorted(os.listdir(folder)):
                if file_name.endswith(".png"):
                    self.data_list.append((os.path.join(folder, file_name), self.class_to_idx[name]))
        self.targets = [label for _, label in self.data_list]

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        path, label = self.data_list[idx]
        with Image.open(path) as im:
            arr = np.array(im)                                           # (28, 28) uint8
        x = torch.tensor(arr, dtype=torch.float32).unsqueeze(0) / 255  # (1, 28, 28)
        if self.transform is not None:
            x = self.transform(x)
        return x, label


train_val_ds = ImageDataset(TRAIN_DIR)
test_ds = ImageDataset(TEST_DIR)
x, label = train_val_ds[0]
print(train_val_ds.classes)
print(len(train_val_ds), len(test_ds), x.shape, x.dtype, label)

In [ ]:
need(train_val_ds.classes == sorted(CLASS_NAMES), f"classes={train_val_ds.classes}")
need(train_val_ds.class_to_idx == {n: i for i, n in enumerate(sorted(CLASS_NAMES))}, "class_to_idx не по порядку classes")
need(test_ds.class_to_idx == train_val_ds.class_to_idx, "у train и test должно быть одно соответствие класс → индекс")
need(len(train_val_ds) == 60000 and len(test_ds) == 10000, f"длины {len(train_val_ds)}, {len(test_ds)}; ждали 60000 и 10000")
need(isinstance(train_val_ds.data_list[0][0], str), "в data_list — пути, а не картинки")
need(len(train_val_ds.targets) == len(train_val_ds), "targets — по одной метке на пример")
need(train_val_ds.targets[:5] == [lbl for _, lbl in train_val_ds.data_list[:5]], "targets в том же порядке, что data_list")

need_tensor(x, "x")
need(x.shape == (1, 28, 28) and x.dtype == torch.float32, f"x: {tuple(x.shape)}, {x.dtype}; ждали (1, 28, 28), float32")
need(0.0 <= x.min().item() and x.max().item() <= 1.0, "значения в 0..1")
need(isinstance(label, int), f"label — int, а не {type(label).__name__}")

_names = train_val_ds.classes
for _i in torch.randint(0, len(train_val_ds), (200,)).tolist():
    _path = train_val_ds.data_list[_i][0]
    _folder = os.path.basename(os.path.dirname(_path))
    need(_names[train_val_ds[_i][1]] == _folder, f"пример {_i}: метка не совпала с папкой {_folder}")
print("5.1 — верно")

## 5.2 Баланс классов
*видео 5: 4:44–5:45*

Автор смотрит на число картинок в каждом классе MNIST и видит лёгкий перекос. Посчитай то же для FashionMNIST через `train_val_ds.targets` (`torch.bincount`), без чтения файлов.

`counts` — словарь имя класса → число картинок; `is_balanced` — `True` / `False`.

**Ожидаемый вывод:**

```text
{'ankle-boot': 6000, 'bag': 6000, 'coat': 6000, ..., 'trouser': 6000} True
```

In [ ]:
_n = torch.bincount(torch.tensor(train_val_ds.targets), minlength=len(train_val_ds.classes))
counts = {name: _n[i].item() for i, name in enumerate(train_val_ds.classes)}
is_balanced = True   # ровно по 6000 в каждом
print(counts, is_balanced)

In [ ]:
need(isinstance(counts, dict) and set(counts) == set(CLASS_NAMES), "ключи counts — имена классов")
need(all(v == 6000 for v in counts.values()), f"counts={counts}")
need(is_balanced is True, "по 6000 в каждом классе — идеальный баланс (в отличие от MNIST у автора)")
print("5.2 — верно")

## 5.3 train / val и три `DataLoader`
*видео 5: 14:07–15:41*

Как у автора: от 60 000 train отрезаем val — 48 000 / 12 000, seed 42. Размер батча — 64 (у автора 16; посмотри, как это изменит число батчей).

Сначала **посчитай в уме** `pred`: пары `(число_батчей, размер_последнего)` для train, val, test. Проверка сравнит с `len(loader)` — без прохода по 70 000 файлам.

**Ожидаемый вывод:**

```text
48000 12000 torch.Size([64, 1, 28, 28]) torch.Size([64])
```

In [ ]:
train_ds, val_ds = random_split(train_val_ds, [48000, 12000], generator=torch.Generator().manual_seed(42))

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

pred = {
    "train": (750, 64),   # 48000 / 64 = 750 ровно
    "val": (188, 32),     # 187 полных + 12000 - 187 * 64 = 32
    "test": (157, 16),    # 156 полных + 10000 - 156 * 64 = 16
}

xb, yb = next(iter(train_loader))
print(len(train_ds), len(val_ds), xb.shape, yb.shape)

In [ ]:
need(len(train_ds) == 48000 and len(val_ds) == 12000, f"длины {len(train_ds)}, {len(val_ds)}")
need(not (set(train_ds.indices) & set(val_ds.indices)), "train и val пересекаются")
_again, _ = random_split(train_val_ds, [48000, 12000], generator=torch.Generator().manual_seed(42))
need(list(_again.indices[:10]) == list(train_ds.indices[:10]), "нужен generator с seed 42")

need(isinstance(train_loader.sampler, RandomSampler), "train — shuffle=True")
need(isinstance(val_loader.sampler, SequentialSampler) and isinstance(test_loader.sampler, SequentialSampler),
     "val и test — без перемешивания")
for _name, _loader in {"train": train_loader, "val": val_loader, "test": test_loader}.items():
    need(_loader.batch_size == BATCH_SIZE, f"{_name}: batch_size={_loader.batch_size}")
    _n = len(_loader.dataset)
    _real = (len(_loader), _n - (len(_loader) - 1) * BATCH_SIZE)
    need(pred[_name] == _real, f"{_name}: ты предсказал {pred[_name]}, на деле {_real}")
need(xb.shape == (64, 1, 28, 28), f"xb.shape={tuple(xb.shape)}, ждали (64, 1, 28, 28)")
need(yb.shape == (64,) and yb.dtype == torch.int64, f"yb: {tuple(yb.shape)}, {yb.dtype}")
print("5.3 — верно")

## 5.4 Посмотреть глазами
*видео 5: 13:03–14:07*

Автор проверяет датасет картинкой: совпадает ли подпись с изображением. Сделай `idx_to_class` (индекс → имя), ячейка ниже нарисует 16 картинок из батча `xb` с подписями. Проверь глазами, что подписи правильные: так ловятся ошибки, которые `assert` не видит.

**Ожидаемый вывод:**

```text
(картинка: 2 ряда по 8 вещей, над каждой — название класса)
5.4 — верно
```

In [ ]:
idx_to_class = {i: name for name, i in train_val_ds.class_to_idx.items()}

In [ ]:
need(idx_to_class == dict(enumerate(train_val_ds.classes)), f"idx_to_class={idx_to_class}")

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, img, lbl in zip(axes.flat, xb[:16], yb[:16]):
    ax.imshow(img[0], cmap="gray")       # img: (1, 28, 28) → img[0]: (28, 28)
    ax.set_title(idx_to_class[lbl.item()], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()
print("5.4 — верно")

**Итог раздела «Видео 5»:** есть `train_loader`, `val_loader`, `test_loader`, батч `(64, 1, 28, 28)` + метки `(64,)`.

**Видео 6** добавит сюда: тот же датасет одной строкой через `ImageFolder` и свою регрессию — картинки с кругом, предсказать центр и радиус.